## 1. Current work directory

In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import os
print(os.getcwd())

In [ ]:
import sys, numpy as np
print("Python:", sys.executable)
print("NumPy:", np.__version__, np.__file__)

## 2. Import packages and define input/output path 

In [ ]:
import glob
import time
import pickle
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from interface_analyzer import analyze_cfm, plot_cfm_k2_single
from interface_analyzer import PTMModifier, analyze_by_custom_modifier, CSPModifier

## 4. Analysis results and plot k and k^2
### Function: `plot_cfm_k2_single(filename, ...)`

This function is designed to analyze the final Capillary Fluctuation Method (CFM) data (which should be stored in a `.dat` file containing $k^2$ vs $k_B T / (L_x L_y \langle|A(k)|^2\rangle)$). The slope of this linear fit directly relates to the interface stiffness ($\tilde{\gamma}$).

#### Key Functionality:

1.  **Optimal Range Selection:** The function iteratively fits the low-$k^2$ data points and selects the subset of points ($n$) that yields the **maximum Coefficient of Determination ($R^2$)**. This provides an objective measure for determining the most linear region of the CFM spectrum.
2.  **Linear Fitting:** Performs a linear fit ($y = m x + b$ or $y = m x$ if `through_origin=True`) on the selected data range.
3.  **Visualization:** Generates a plot showing all data points, highlighting the points used for the best fit, and displaying the resulting fit line.

#### Key Parameters:

| Parameter | Description |
| :--- | :--- |
| **`filename`** | Path to the input `.dat` file (must contain $k^2$, $\text{Ak}_{min}$, and $\text{Ak}_{max}$). |
| **`k2_min`** | Minimum $k^2$ value to consider for the fit. Filters out the first point which is usually zero. |
| **`min_points`** | Minimum number of data points required to perform the linear fit. |
| **`L_min_interface`** | Defines the maximum $k^2$ cutoff based on the expected minimum interface width ($\sim (2\pi / (L_{\text{min}} \cdot a))^2$). |
| **`through_origin`** | If `True`, forces the linear regression line to pass through the origin ($b=0$). |

#### Return Value:

Returns a dictionary containing the calculated fit results, including the calculated `slope` (interface stiffness), `intercept`, maximum `r2`, and the $k^2$ range (`k2_min_used`, `k2_max_used`).

In [ ]:
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# User settings
# =========================
SAVE_DIR = FULL_DATA_ROOT / "100_010_LOP_smoothing_parameter"
TEMPERATURE_K = 925.08
LATTICE_CONST_A = 4.045

# fitting settings
K2_MIN = 5.0e-3
L_MIN_INTERFACE = 4
MIN_POINTS = 6
THROUGH_ORIGIN = True

# =========================
# Helper: parse a_grid and d from filename
# Example:
# 100_010_cfg_post_orientation_grid_2_5_ang_part_d_4_0.pkl
# =========================
def parse_params_from_name(path_obj):
    name = path_obj.name
    m = re.match(
        r"100_010_cfg_post_orientation_grid_(\d+_\d+)_ang_part_d_(\d+_\d+)\.pkl",
        name
    )
    if not m:
        return None, None
    a_grid = float(m.group(1).replace("_", "."))
    d = float(m.group(2).replace("_", "."))
    return a_grid, d

def fmt_sig3(x):
    """Format with 3 significant digits."""
    return f"{x:.3g}"

# =========================
# Collect all pkl files
# =========================
pkl_files = sorted(
    SAVE_DIR.glob("100_010_cfg_post_orientation_grid_*_ang_part_d_*.pkl")
)

if not pkl_files:
    raise FileNotFoundError("No matching pkl files found in current directory.")

summary_rows = []

for pkl_path in pkl_files:
    a_grid, d = parse_params_from_name(pkl_path)
    if a_grid is None:
        print(f"Skipping unrecognized filename: {pkl_path.name}")
        continue

    tag = f"grid_{a_grid:.1f}_d_{d:.1f}".replace(".", "_")
    output_base = SAVE_DIR / f"ptm_cfm_output_{tag}"

    print(f"\n=== Processing {pkl_path.name} ===")
    print(f"a_grid = {a_grid:.1f} Å, d = {d:.1f} Å")

    # -------------------------
    # Step 1: analyze CFM
    # -------------------------
    results_ptm = analyze_cfm(
        pickle_path=pkl_path,
        T=TEMPERATURE_K,
        a=LATTICE_CONST_A,
        use_pchip=True,
        pchipres=10000,
        show_plot=True
    )

    # Save analyze_cfm figure if generated
    plt.savefig(str(output_base) + "_analyze_cfm.png", dpi=300, bbox_inches="tight")
    plt.close()

    # -------------------------
    # Step 2: save k^2 data
    # -------------------------
    k2 = results_ptm["k2"]
    Ak_min = results_ptm["Ak_min"]
    Ak_max = results_ptm["Ak_max"]

    intdata2 = np.c_[k2, Ak_min, Ak_max]

    k2_dat_path = str(output_base) + "_k2.dat"
    np.savetxt(
        k2_dat_path,
        intdata2,
        fmt="%.8e",
        header="k^2 Ak_min Ak_max"
    )
    print(f"CFM data saved to: {k2_dat_path}")

    # -------------------------
    # Step 3: linear fit
    # -------------------------
    res_fit = plot_cfm_k2_single(
        k2_dat_path,
        label=f"a_grid={a_grid:.1f} Å, d={d:.1f} Å",
        k2_min=K2_MIN,
        L_min_interface=L_MIN_INTERFACE,
        min_points=MIN_POINTS,
        through_origin=THROUGH_ORIGIN
    )

    # Save linear fit plot
    plt.savefig(str(output_base) + "_fit.png", dpi=300, bbox_inches="tight")
    plt.close()

    slope = res_fit["slope"]
    slope_sig3 = fmt_sig3(slope)

    print("--- Linear Fit Results ---")
    print(res_fit)
    print(f"slope (3 sig. figs.) = {slope_sig3}")

    summary_rows.append({
        "pkl_file": pkl_path.name,
        "a_grid_A": a_grid,
        "d_A": d,
        "slope": slope,
        "slope_3sig": slope_sig3
    })

# =========================
# Save summary
# =========================
df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.sort_values(by=["a_grid_A", "d_A"]).reset_index(drop=True)

summary_csv = SAVE_DIR / "cfm_slope_summary.csv"
df_summary.to_csv(summary_csv, index=False)

print("\n===================================")
print("All jobs finished.")
print(f"Summary saved to: {summary_csv}")
print(df_summary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract data
df = df_summary.copy()

# Group by a_grid
a_grids = sorted(df["a_grid_A"].unique())

# Define colors and markers
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
markers = ["o", "s", "^", "D","<"]

plt.figure(figsize=(7, 5))

for i, a in enumerate(a_grids):
    sub = df[df["a_grid_A"] == a].sort_values("d_A")

    d_vals = sub["d_A"].values
    slopes = sub["slope"].values

    plt.plot(
        d_vals,
        slopes,
        linestyle="-",
        linewidth=1.8,
        color=colors[i % len(colors)],
        marker=markers[i % len(markers)],
        markersize=7,
        label=f"a_grid = {a:.1f} Å"
    )

plt.xlabel("Smoothing radius d (Å)")
plt.ylabel("Interfacial stiffness (slope)")
plt.title("Sensitivity of interfacial stiffness to smoothing radius")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("stiffness_vs_d.png", dpi=300)
plt.show()

## 5. Check the volume fraction of solid phase
CFM needs a relatively stable interface position, therefore the fraction of solid phase should be a constant.

In [ ]:
from matplotlib import pyplot as plt
with open(Path_ptm_pkl_d4, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)
frames = sorted(results_all.keys())
mean_upper = [results_all[i]["h_upper"].mean() for i in frames]
mean_lower = [results_all[i]["h_lower"].mean() for i in frames]
solid = np.asarray(mean_upper) - np.asarray(mean_lower)

# Plot
plt.figure(figsize=(6,4))
plt.plot(frames, solid, 'o-', lw=1.5, markersize=4)
plt.xlabel("Frame index (timestep)")
plt.ylabel("Solid phase length (Å)")
plt.title("Evolution of solid-liquid interface position")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(solid,bins=20,label="100_010")
plt.title("Histogram of Solid Volume")
plt.legend()
plt.show()

In [ ]:
from matplotlib import pyplot as plt
with open(Path_ptm_pkl_d8, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)
frames = sorted(results_all.keys())
mean_upper = [results_all[i]["h_upper"].mean() for i in frames]
mean_lower = [results_all[i]["h_lower"].mean() for i in frames]
solid = np.asarray(mean_upper) - np.asarray(mean_lower)

# Plot
plt.figure(figsize=(6,4))
plt.plot(frames, solid, 'o-', lw=1.5, markersize=4)
plt.xlabel("Frame index (timestep)")
plt.ylabel("Solid phase length (Å)")
plt.title("Evolution of solid-liquid interface position")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(solid,bins=20,label="100_010")
plt.title("Histogram of Solid Volume")
plt.legend()
plt.show()

In [ ]:
from matplotlib import pyplot as plt
from pathlib import Path
import pickle
SAVE_DIR = FULL_DATA_ROOT
Path_ptm_pkl_a_2p5_d_6 = SAVE_DIR / "100_010_cfg_post_orientation_grid_2_5_ang_part_d_6_0.pkl"
with open(Path_ptm_pkl_a_2p5_d_6, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)
Phase = results_all[5000000]["M"]

plt.figure(figsize=(10, 8))
plt.imshow(Phase, cmap='viridis', origin='lower')
plt.colorbar(label='Phase Value', shrink=0.5)
plt.xlabel('X index')
plt.ylabel('Y index')
#plt.savefig("phase_heatmap_PTM_d4.png")
plt.show()

In [ ]:
Boundary = results_all[5000000]["h_lower"]
plt.figure(figsize=(8, 2.0))
plt.plot(Boundary)
plt.xlabel('X index')
plt.ylabel('Y index')
plt.show()

In [ ]:
import re
import pickle
import math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

SAVE_DIR = FULL_DATA_ROOT
FRAME_ID = 5000000

def parse_params_from_name(path_obj):
    name = path_obj.name
    m = re.match(
        r"100_010_cfg_post_orientation_grid_(\d+_\d+)_ang_part_d_(\d+_\d+)\.pkl",
        name
    )
    if not m:
        return None, None
    a_grid = float(m.group(1).replace("_", "."))
    d = float(m.group(2).replace("_", "."))
    return a_grid, d

# Collect all pkl files
pkl_files = sorted(
    SAVE_DIR.glob("100_010_cfg_post_orientation_grid_*_ang_part_d_*.pkl")
)

# Read data
all_data = []
for pkl_path in pkl_files:
    a_grid, d = parse_params_from_name(pkl_path)
    if a_grid is None:
        continue

    with open(pkl_path, "rb") as f:
        results_all = pickle.load(f)

    if FRAME_ID not in results_all:
        print(f"Frame {FRAME_ID} not found in {pkl_path.name}")
        continue

    phase = results_all[FRAME_ID]["M"]
    h_lower = results_all[FRAME_ID]["h_lower"]

    all_data.append({
        "file": pkl_path.name,
        "a_grid": a_grid,
        "d": d,
        "M": phase,
        "h_lower": h_lower
    })

# Sort by a_grid first, then d
all_data = sorted(all_data, key=lambda x: (x["a_grid"], x["d"]))

if not all_data:
    raise ValueError("No valid data found.")

# ---------- Plot Phase subplots ----------
n = len(all_data)
ncols = 4
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.6*nrows), constrained_layout=True)
axes = np.array(axes).reshape(-1)

# Use a shared color scale for easier comparison
vmin = min(np.min(item["M"]) for item in all_data)
vmax = max(np.max(item["M"]) for item in all_data)

im = None
for ax, item in zip(axes, all_data):
    im = ax.imshow(
        item["M"],
        cmap="viridis",
        origin="lower",
        aspect="auto",
        vmin=vmin,
        vmax=vmax
    )
    ax.set_title(f"a_grid={item['a_grid']:.1f} Å, d={item['d']:.1f} Å")
    ax.set_xlabel("X index")
    ax.set_ylabel("Y index")

# Turn off unused subplots
for ax in axes[len(all_data):]:
    ax.axis("off")

cbar = fig.colorbar(im, ax=axes[:len(all_data)], shrink=0.85)
cbar.set_label("Phase value")

out_path = SAVE_DIR / f"phase_heatmap_compare_frame_{FRAME_ID}.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {out_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 4.5))

for item in all_data:
    h = np.asarray(item["h_lower"], dtype=float)
    h_centered = h - np.mean(h)

    dx = item["a_grid"]  # grid spacing
    x = np.arange(len(h)) * dx  # Convert to physical coordinates (Å)

    plt.plot(
        x,
        h_centered,
        linewidth=1.5,
        label=f"a_grid={item['a_grid']:.1f}, d={item['d']:.1f}"
    )

plt.xlabel("x (Å)")
plt.ylabel(r"$h(x)-\langle h\rangle$")
plt.title("Interface shape comparison (physical coordinate)")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()

out_path = SAVE_DIR / "h_lower_centered_compare_physical.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {out_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Combinations to select
target_pairs = [
    (2.0, 4.0),
    (2.0, 10.0),
    (3.0, 4.0),
    (3.0, 10.0),
]

plt.figure(figsize=(10, 4.5))

for item in all_data:
    a = item["a_grid"]
    d = item["d"]

    if (a, d) not in target_pairs:
        continue

    h = np.asarray(item["h_lower"], dtype=float)
    h_centered = h - np.mean(h)

    dx = a
    x = np.arange(len(h)) * dx  # Physical coordinate

    plt.plot(
        x,
        h_centered,
        linewidth=2.0,
        label=f"a={a:.1f} Å, d={d:.1f} Å"
    )

plt.xlabel("x (Å)")
plt.ylabel(r"$h(x)-\langle h\rangle$")
plt.title("Interface shape comparison (selected cases)")
plt.legend()
plt.tight_layout()

out_path = SAVE_DIR / "h_lower_compare_selected_physical.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {out_path}")

In [ ]:
from matplotlib import pyplot as plt
with open(Path_ptm_pkl_d8, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)
Phase = results_all[5000000]["M"]

plt.figure(figsize=(10, 8))
plt.imshow(Phase, cmap='viridis', origin='lower')
plt.colorbar(label='Phase Value', shrink=0.5)
plt.xlabel('X index')
plt.ylabel('Y index')
plt.savefig("phase_heatmap_PTM_d4.png")
plt.show()

In [ ]:
Boundary = results_all[5000000]["h_lower"]
plt.figure(figsize=(8, 2.0))
plt.plot(Boundary)
plt.xlabel('X index')
plt.ylabel('Y index')
plt.show()

In [ ]:
plt.plot(Phase[:,50])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def find_interface_positions(Phase, criteria, axis=0):
    """
    For a 2D Phase map, find where each line cut intersects the criteria value along the specified direction.
    
    Parameters
    ----------
    Phase : 2D np.ndarray
        phase-parameter field
    criteria : float
        threshold; values greater than criteria are liquid and values below criteria are solid
    axis : int
        Axis along which to locate the interface:
        - axis=0: find the interface for each column Phase[:, j] (usually the current case)
        - axis=1: find the interface for each row Phase[i, :]
    
    Returns
    -------
    interface_pos : 1D np.ndarray
        interface position for each line cut; np.nan if no crossing is found
    """
    if axis == 1:
        Phase = Phase.T  # transpose for unified handling so that operations always use Phase[:, j]

    n_normal, n_parallel = Phase.shape
    interface_pos = np.full(n_parallel, np.nan)

    for j in range(n_parallel):
        profile = Phase[:, j]
        diff = profile - criteria

        # Find sign changes, which indicate crossings of criteria
        sign_change_idx = np.where(diff[:-1] * diff[1:] <= 0)[0]

        if len(sign_change_idx) == 0:
            continue

        # If multiple crossings are present, use the first one by default
        i = sign_change_idx[0]

        y1 = profile[i]
        y2 = profile[i + 1]

        # Avoid division by zero
        if y2 == y1:
            x_cross = i
        else:
            # Linear interpolation: solve for the position where profile = criteria
            x_cross = i + (criteria - y1) / (y2 - y1)

        interface_pos[j] = x_cross

    return interface_pos

In [ ]:
criteria = 10.0
interface_pos = find_interface_positions(Phase, criteria, axis=0)

plt.figure(figsize=(10, 8))
plt.imshow(Phase, cmap='viridis', origin='lower', aspect='auto')
plt.colorbar(label='Phase Value', shrink=0.6)

x_parallel = np.arange(len(interface_pos))
plt.plot(x_parallel, interface_pos, 'r-', lw=2, label=f'Interface (criteria={criteria})')

plt.xlabel('Parallel direction index')
plt.ylabel('Normal direction index')
plt.legend()
plt.tight_layout()
plt.savefig("interface_overlay_criteria_10.png", dpi=300)
plt.show()

print(f"criteria = {criteria}")
print(f"mean interface position = {np.nanmean(interface_pos):.4f}")
print(f"std  interface position = {np.nanstd(interface_pos):.4f}")

In [ ]:
criteria_list = [7, 9, 11, 13, 15]

all_interfaces = {}
mean_positions = []
std_positions = []

for c in criteria_list:
    interface_pos = find_interface_positions(Phase, c, axis=0)
    all_interfaces[c] = interface_pos
    mean_positions.append(np.nanmean(interface_pos))
    std_positions.append(np.nanstd(interface_pos))

plt.figure(figsize=(10, 8))
plt.imshow(Phase, cmap='viridis', origin='lower', aspect='auto')
plt.colorbar(label='Phase Value', shrink=0.6)

x_parallel = np.arange(Phase.shape[1])

for c in criteria_list:
    plt.plot(x_parallel, all_interfaces[c], lw=1.8, label=f'c={c}')

plt.xlabel('Parallel direction index')
plt.ylabel('Normal direction index')
plt.legend()
plt.tight_layout()
plt.savefig("interface_overlay_multiple_criteria.png", dpi=300)
plt.show()

In [ ]:
Boundary = results_all[3000000]["h_lower"]
plt.figure(figsize=(8, 2.0))
plt.plot(Boundary)
plt.xlabel('X index')
plt.ylabel('Y index')
plt.savefig("Interface_LOP_vector1_5.png")
plt.show()

In [ ]:
import numpy as np

# 1. Collect and sort all timesteps to keep the time series ordered
timesteps = sorted(results_all.keys())

# 2. Use a list comprehension to extract h_lower (or h_upper) for each frame
# Assume each results_all[ts]["h_lower"] is an array or Series of length N_bins
h_lower_list = [results_all[ts]["h_lower"] for ts in timesteps]

# 3. Convert the list to a 2D NumPy array with shape (N_frames, N_bins)
h_lower_series = np.array(h_lower_list)

# 4. Check the shape
print(f"Number of frames (N_frames): {h_lower_series.shape[0]}")
print(f"Number of bins (N_bins): {h_lower_series.shape[1]}")

# This array can now be passed to the analysis function
# mu_slope = calculate_fluctuation_kinetics(h_lower_series, dt=..., Lx=...)

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

def calculate_advanced_mu(h_series, dt, Lx, Tm, latent_heat_vol, stiffness):
    """
    Compute the kinetic coefficient mu following [cite: 448-472], keeping the units self-consistent.
    
    Recommended unit system (LAMMPS metal units):
    --------------------------------
    h_series        : np.array (N_frames, N_bins), unit: [Angstrom]
    dt              : frame interval, unit: [ps]
    Lx              : interface box length, unit: [Angstrom]
    Tm              : melting point, unit: [K]
    latent_heat_vol : latent heat per unit volume (L), unit: [eV/Angstrom^3]
                      Note: if the input data are in J/m^3, divide by 1.60218e28
    stiffness       : interface stiffness (gamma + gamma''), unit: [eV/Angstrom^2]
                      Note: if the input data are in mJ/m^2 (or erg/cm^2), divide by 1.60218e3
    
    Returns:
    --------------------------------
    mu_SI           : kinetic coefficient, unit: [m/(s·K)]
    """
    
    # 1. Compute the Gibbs-Thomson coefficient Gamma (unit: A·K) [cite: 458]
    gamma_gt_MD = (Tm * stiffness) / latent_heat_vol
    
    n_frames, n_bins = h_series.shape
    k_vals = 2 * np.pi * np.fft.rfftfreq(n_bins, d=Lx/n_bins) # unit: A^-1
    amplitudes = np.fft.rfft(h_series - np.mean(h_series, axis=1, keepdims=True), axis=1, norm="forward")
    
    tau_list = []
    k_fit_list = []
    
    def normalized_decay_model(t, tau):
        return 1 - np.exp(-t / tau) # 

    max_lag = n_frames // 5
    time_lags = np.arange(max_lag) * dt # unit: ps

    for i in range(1, len(k_vals)):
        A_k = amplitudes[:, i]
        mean_sq_A = np.mean(np.abs(A_k)**2)
        
        lhs = []
        for lag in range(max_lag):
            diff_sq = np.abs(A_k[lag:] - A_k[:-lag if lag > 0 else None])**2
            lhs.append(np.mean(diff_sq) / (2 * mean_sq_A))
        lhs = np.array(lhs)
        
        try:
            # Fit tau (ps)
            popt, _ = curve_fit(normalized_decay_model, time_lags[lhs > 0.1], lhs[lhs > 0.1], p0=[10.0])
            tau_list.append(popt[0])
            k_fit_list.append(k_vals[i])
        except:
            continue

    k_fit = np.array(k_fit_list)
    tau_fit_ps = np.array(tau_list)
    
    # 2. Extract mu_k (SIunit: m/s/K) [cite: 451, 469]
    inv_tau = 1.0 / tau_fit_ps
    rhs_for_mu = gamma_gt_MD * (k_fit**2)
    mu_MD, _ = np.polyfit(rhs_for_mu, inv_tau, 1)
    mu_SI = mu_MD * 100.0 

    # --- 3. Plot the results (matching the coordinate range in the manuscript) ---
    # Convert tau to ns before plotting so that it falls in the 0.0008-0.2 range 
    tau_fit_ns = tau_fit_ps / 1000.0 
    
    plt.figure(figsize=(8, 6))
    
    # Plot simulation points
    plt.loglog(k_fit, tau_fit_ns, 'o', color='tab:blue', label='Simulation Data', markersize=7)
    
    # Plot the fitted line: tau(ns) = 1 / (mu_MD * Gamma * k^2 * 1000)
    tau_theory_ns = 1.0 / (mu_MD * gamma_gt_MD * k_fit**2 * 1000.0)
    plt.loglog(k_fit, tau_theory_ns, '-', color='tab:orange', 
               label=f'Fit: $\\mu_k$ = {mu_SI:.3f} m/s/K')
    
    # Set the requested plotting range
    plt.xlim(0.02, 0.4)   # x-axis range: 0.02 to 0.4 [A^-1]
    plt.ylim(0.0008, 0.2) # y-axis range: 0.0008 to 0.2 [ns]
    
    plt.xlabel(r'$k$ ($\AA^{-1}$)', fontsize=12)
    plt.ylabel(r'$\tau$ (ns)', fontsize=12)
    plt.title('Kinetic Coefficient Fitting (Fluctuation Spectra)', fontsize=13)
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    return mu_SI

def calculate_kinetic_anisotropy(mu_max, mu_min):
    """Compute the kinetic anisotropy parameter epsilon_k """
    return (mu_max - mu_min) / (mu_max + mu_min)

In [ ]:
latent_heat_vol_MD = 9.4e8 / 1.60218e11 # J/m^3 => ev/A^3
stiffness_MD = 87 / 1.60218e4 # mJ/m^2 => ev/A^2
mu_k = calculate_advanced_mu(h_lower_series, dt=0.5, Lx=584.18, Tm=925, latent_heat_vol=latent_heat_vol_MD, stiffness=stiffness_MD)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# --- Parameter settings; adjust for the simulation being analyzed ---
Lx = 584.18          # Box length in the X direction [Angstrom]
dt = 0.5              # trajectory output interval [ps]
Tm = 925.0            # melting point [K]
L_vol = latent_heat_vol_MD       # latent heat per unit volume [eV/A^3]
stiffness = stiffness_MD   # interface stiffness [eV/A^2]

# --- 1. Merge and extract data ---
timesteps = sorted(results_all.keys())
h_upper_all = np.array([results_all[ts]["h_upper"] for ts in timesteps])
h_lower_all = np.array([results_all[ts]["h_lower"] for ts in timesteps])
interfaces = [h_upper_all, h_lower_all]

N_frames, N_bins = h_upper_all.shape
k_vals = 2 * np.pi * np.fft.rfftfreq(N_bins, d=Lx/N_bins) # [A^-1]

# Perform the Fourier transform
amplitudes_list = [np.fft.rfft(h - np.mean(h, axis=1, keepdims=True), axis=1, norm="forward") for h in interfaces]

print(f"Data loaded: {N_frames} frames, {N_bins} bins. Extracted Fourier amplitudes for the two interfaces.")

In [ ]:
# --- 2. Fit settings ---
max_lag = N_frames // 5
time_lags_ns = np.arange(max_lag) * dt / 1000.0
tau_results = []
k_results = []

def decay_model(t, tau):
    return 1 - np.exp(-t / tau) # [cite: 451]

plt.figure(figsize=(6, 6))

# Loop over the specified index range: 5 to 13 (inclusive)
for i in range(5, 15):
    # Joint averaging: combine statistics from the two interfaces
    sum_diff_sq = np.zeros(max_lag)
    sum_total_power = 0
    
    for amp in amplitudes_list:
        A_k = amp[:, i]
        sum_total_power += 2 * np.mean(np.abs(A_k)**2) # 2 * <|A(k,0)|^2> [cite: 451, 483]
        for lag in range(max_lag):
            diff = np.abs(A_k[lag:] - A_k[:-lag if lag > 0 else None])**2
            sum_diff_sq[lag] += np.mean(diff) # <|A(k,t)-A(k,0)|^2> [cite: 451, 483]
            
    # Compute the normalized correlation function LHS
    lhs = sum_diff_sq / sum_total_power
    is_too_high = (lhs >= 0.96)
    has_reached_cutoff = np.maximum.accumulate(is_too_high)
    # Set the fitting mask; use only points between 0.7 and 0.96
    fit_mask = (~has_reached_cutoff) & (lhs > 0.5)
    not_fit_mask = ~fit_mask
    if np.sum(fit_mask) > 3: # Ensure that enough points are available for fitting
        try:
            # Fit and extract tau [ns] [cite: 468]
            popt, _ = curve_fit(decay_model, time_lags_ns[fit_mask], lhs[fit_mask], p0=[0.01])
            tau_val = popt[0]
            tau_results.append(tau_val)
            k_results.append(k_vals[i])
            
            # Plot to inspect fit quality
            color = plt.cm.plasma((i-5)/9)
            plt.scatter(time_lags_ns[not_fit_mask], lhs[not_fit_mask], 
                        s=10, facecolors='none', edgecolors=color, alpha=0.3)

            # 2. Plot the selected points as filled markers 
            plt.scatter(time_lags_ns[fit_mask], lhs[fit_mask], 
                        s=15, color=color, alpha=0.8)
            plt.plot(time_lags_ns, decay_model(time_lags_ns, tau_val), color=color, 
                     label=f'k={k_vals[i]:.4f}, $\\tau$={tau_val:.4f}ns')
        except:
            print(f"Index {i} (k={k_vals[i]:.4f}) fit failed")

plt.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
plt.axhline(0.96, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Time (ns)')
plt.ylabel('Normalized Correlation Function')
plt.title('110[1-10]')
plt.xlim(-0.002, 0.1)
plt.legend(loc='best')
plt.grid(True, alpha=0.2)
# plt.savefig("110_1-10_decay.svg",format="svg")
plt.show()

In [ ]:
# --- 3. Extract the kinetic coefficient mu ---
k_array = np.array(k_results)
tau_array = np.array(tau_results)
gamma_gt = (Tm * stiffness) / L_vol # [A*K] [cite: 458]

# Linear fitting: 1/tau = (mu * Gamma) * k^2 -> mu = 1 / (tau * Gamma * k^2) [cite: 451, 452]
inv_tau = 1.0 / tau_array
rhs = gamma_gt * (k_array**2)
mu_MD, _ = np.polyfit(rhs, inv_tau, 1) # the slope is mu [A/(ns*K)]
mu_SI = mu_MD * 0.1 # Convert A/ns to m/s; result unit is m/s/K

print(f"Fitted kinetic coefficient mu: {mu_SI:.4f} m/s/K")

# --- Plot the results ---
plt.figure(figsize=(7, 6))
plt.loglog(k_array, tau_array, 'ks', label='MD Results (Combined)')

# Plot the reference line: tau = 1 / (mu * Gamma * k^2)
k_ref = np.linspace(0.02, 0.4, 100)
tau_ref = 1.0 / (gamma_gt * mu_MD * k_ref**2)
plt.loglog(k_ref, tau_ref, 'r-', label=f'Fit Line ($\mu$={mu_SI:.3f} m/s/K)')

plt.xlim(0.02, 0.4)   # range used in the manuscript [cite: 472]
plt.ylim(0.0008, 0.2) # range used in the manuscript [cite: 472]
plt.xlabel(r'$k$ ($\AA^{-1}$)')
plt.ylabel(r'$\tau$ (ns)')
plt.title('110[1-10]')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.legend()
# plt.savefig("110_1-10_mu_k_fitting.svg",format="svg")
plt.show()

## 6. Variation of the slope with respect to the k-range
### Function: `analyze_cfm_fit_sensitivity(filename, ...)`

#### Interface Stiffness Sensitivity Analysis

The choice of the low-$k^2$ fitting range is often subjective and critical to the final calculated interface stiffness ($\tilde{\gamma}$). This function quantifies the robustness of the result by analyzing the fit parameters as a function of the data range.

It systematically iterates through all valid low-$k^2$ data subsets (starting from the minimum number of points, `min_points`, up to the maximum filtered dataset size) and performs a linear fit for each subset.

#### Key Output:

The function generates two plots:

1.  **Interface Stiffness (Slope $m$) vs. Number of $k^2$ Points ($n$):** Shows how sensitive the stiffness value is to the inclusion of higher-$k^2$ data points.
2.  **Coefficient of Determination ($R^2$) vs. Number of $k^2$ Points ($n$):** Identifies the range where the linear relationship is strongest (highest $R^2$).

#### Return Value:

Returns a dictionary containing arrays for:
* `n_points`: The number of data points used in each successive fit.
* `stiffness`: The slope ($m$) calculated for each fit.
* `r2`: The $R^2$ value calculated for each fit.

In [ ]:
from interface_analyzer import analyze_cfm_fit_sensitivity

fit_results = analyze_cfm_fit_sensitivity(
    "ptm_cfm_output_k2.dat",
    k2_min=5e-3,
    min_points=5,
    L_min_interface=4,
    through_origin=True
)

# The plots will be displayed, and fit_results will contain arrays
# for 'n_points', 'stiffness', and 'r2'.
print("Stiffness values calculated:", fit_results['stiffness'])
csp_fit_output = "csp_cfm_fit_sensitivity.dat"
data_matrix = np.c_[fit_results['n_points'], fit_results['stiffness'], fit_results['r2']]

# 2. Define the header
header = "N_Points_Used | Stiffness_Slope | R2_Coefficient"
np.savetxt(
        csp_fit_output,
        data_matrix,
        fmt="%d %.8e %.8f",  # Format: integer, scientific notation, fixed float
        header=header,
        comments='# '        # Ensures header is commented out
    )

In [ ]:
from interface_analyzer import analyze_cfm_fit_sensitivity
fit_results = analyze_cfm_fit_sensitivity(
    ptm_k2_path,
    k2_min=10.0e-4,
    min_points=5,
    L_min_interface=4,
    through_origin=True
)

# The plots will be displayed, and fit_results will contain arrays
# for 'n_points', 'stiffness', and 'r2'.
print("Stiffness values calculated:", fit_results['stiffness'])
ptm_fit_output = "ptm_cfm_fit_sensitivity.dat"
data_matrix = np.c_[fit_results['n_points'], fit_results['stiffness'], fit_results['r2']]

# 2. Define the header
header = "N_Points_Used | Stiffness_Slope | R2_Coefficient"
np.savetxt(
        ptm_fit_output,
        data_matrix,
        fmt="%d %.8e %.8f",  # Format: integer, scientific notation, fixed float
        header=header,
        comments='# '        # Ensures header is commented out
    )